In [ ]:
import requests
import math
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

def get_api_key():
    """
    Securely load and validate Ticketmaster API key.
    KEY IS NOT STORED IN MEMORY - only loaded when called.
    
    Returns:
        str: Validated API key
    
    Raises:
        ValueError: If key not found or invalid format
    """
    api_key = os.getenv('TICKETMASTER_API_KEY')
    
    if not api_key:
        raise ValueError(
            "❌ TICKETMASTER_API_KEY not found in .env file\n"
            "Please create a .env file from .env.example and add your API key"
        )
    
    # Validate format (Ticketmaster keys are typically 32+ alphanumeric characters)
    if len(api_key) < 20 or not api_key.isalnum():
        raise ValueError(
            "❌ TICKETMASTER_API_KEY appears invalid\n"
            "Ensure it's properly formatted in your .env file"
        )
    
    return api_key

# Test that API key loads successfully (but don't store it)
try:
    test_key = get_api_key()
    print("✅ API key loaded successfully")
except ValueError as e:
    print(e)
    raise

In [2]:
## My files structures

In [3]:
# example of how to keep track of artist ids where I've already been notified.
{
  "G1vZ917Xy1": {
    "name": "Taylor Swift Concert",
    "date": "2025-12-01",
    "venue": "Madison Square Garden",
    "city": "New York",
    "notify_count": 1,
    "last_notified": "2025-11-16T12:00:00"
  },
  "H8xY123Abc": {
    "name": "The Weeknd Live",
    "date": "2025-12-05",
    "venue": "United Center",
    "city": "Chicago",
    "notify_count": 0,
    "last_notified": 'null'
  }
}


{'G1vZ917Xy1': {'name': 'Taylor Swift Concert',
  'date': '2025-12-01',
  'venue': 'Madison Square Garden',
  'city': 'New York',
  'notify_count': 1,
  'last_notified': '2025-11-16T12:00:00'},
 'H8xY123Abc': {'name': 'The Weeknd Live',
  'date': '2025-12-05',
  'venue': 'United Center',
  'city': 'Chicago',
  'notify_count': 0,
  'last_notified': 'null'}}

In [4]:
my_settings = {
    "artists": [
        "Peter McPoland",
        "Jonah Kagen",
        "The Neighbourhood"
    ],
    "location": {
        "Madison": {"state": "WI", "dmaId": "338"},
        "Milwaukee": {"state": "WI", "dmaId": "349"},
        "Chicago": {"state": "IL", "dmaId": "Chicago_DMA_ID"},
        'travel_info' : {
            "latlong": "43.0731,-89.4012",    # Madison, WI
            "radius": 300,                     # miles
            "unit": "miles"
        }
    },
    "radius_miles": 25,        # optional for radius searches
    "classificationName": "Music",   # filters only music events
    "genre": None              # optional filter if you want specific genres
}

In [5]:
# V2

In [ ]:
import math
import requests
from datetime import datetime

def compute_pagination(total_events, page_size=25):
    """
    Compute pagination details given total events and page size.
    """
    # TODO: Add error handling (negative numbers, invalid types)
    pages_needed = math.ceil(total_events / page_size)
    return {
        "page_size": page_size,
        "pages_needed": pages_needed
    }

def fetch_all_events(artist_id, travel_info, total_events):
    """
    Fetch all events for a given artist using pagination.
    Returns a list of raw event dictionaries.
    """
    url = "https://app.ticketmaster.com/discovery/v2/events.json"
    page_size = 50  # Ticketmaster max
    pages_needed = (total_events + page_size - 1) // page_size

    all_events = []

    for page in range(pages_needed):
        params = {
            "attractionId": artist_id,
            "latlong": travel_info['latlong'],
            "radius": travel_info['radius'],
            "unit": travel_info['unit'],
            "apikey": get_api_key(),
            "size": page_size,
            "page": page
        }

        # TODO: Add error handling for network issues, non-200, missing _embedded
        response = requests.get(url, params=params)
        data = response.json()
        events = data.get("_embedded", {}).get("events", [])
        all_events.extend(events)

    return all_events

def building_artist_metadata(artist_name, artist_dict):
    """
    Query Ticketmaster for an artist and store metadata in structured format.
    """
    url = "https://app.ticketmaster.com/discovery/v2/attractions.json"
    params = {"keyword": artist_name, "apikey": get_api_key()}

    # TODO: Add error handling for network failure, 0 attractions, multiple matches
    artist_metadata = requests.get(url, params=params).json()
    
    # Extract first attraction
    attraction = artist_metadata['_embedded']['attractions'][0]
    
    # Get total upcoming events
    total_events = attraction['upcomingEvents']['_total']

    # Compute pagination
    pagination_info = compute_pagination(total_events, page_size=25)

    # Build artist dictionary with multiple IDs
    individual_artist_dict = {
        "ids": [attraction['id']],
        "genre_name": attraction['classifications'][0]['genre']['name'],
        "genre_id": attraction['classifications'][0]['genre']['id'],
        "ticketmaster_upcoming_events_number": total_events,
        "pagination": pagination_info,
        "listening_score": None,  # placeholder for future Apple Music integration
        "next_event_date": None,  # will be populated after fetching events
        "total_notified_events": 0,
        "last_checked": datetime.now().isoformat(),
        "events": []
    }

    artist_dict[artist_name] = individual_artist_dict

def building_artist_event_data(artist_name, artist_dict):
    """
    Fetch all upcoming events for an artist in configured locations and attach them to the artist dict.
    """
    artist_info = artist_dict[artist_name]
    artist_id = artist_info['ids'][0]  # for now just first ID
    total_events = artist_info['ticketmaster_upcoming_events_number']
    travel_info = my_settings['location']['travel_info']

    # Fetch events with pagination
    raw_events = fetch_all_events(artist_id, travel_info, total_events)

    structured_events = []
    next_event_date = None

    for event in raw_events:
        venue_info = event.get("_embedded", {}).get("venues", [{}])[0]

    # Extract min/max price if available
    price_ranges = event.get("priceRanges", [])
    min_price = price_ranges[0].get("min") if price_ranges else None
    max_price = price_ranges[0].get("max") if price_ranges else None

    # Calculate distance if venue coordinates available
    distance_miles = None
    if "location" in venue_info and venue_info["location"]:
        from geopy.distance import geodesic
        my_lat, my_long = map(float, travel_info['latlong'].split(','))
        venue_lat, venue_long = float(venue_info["location"]["latitude"]), float(venue_info["location"]["longitude"])
        distance_miles = geodesic((my_lat, my_long), (venue_lat, venue_long)).miles
        
        event_data = {
            "id": event.get("id"),
            "name": event.get("name"),
            "date": event.get("dates", {}).get("start", {}).get("localDate"),
            "time": event.get("dates", {}).get("start", {}).get("localTime"),
            "venue": venue_info.get("name"),
            "city": venue_info.get("city", {}).get("name"),
            "state": venue_info.get("state", {}).get("stateCode"),
            "ticket_url": event.get("url"),
            "status": event.get("dates", {}).get("status", {}).get("code"),
            "min_price": min_price,
            "max_price": max_price,
            "distance_miles": distance_miles,
            "notified": False,
            "notification_count": 0,
            "notifications": []
        }
        
        structured_events.append(event_data)

        # Track next event date
        event_date = event_data["date"]
        if event_date and (next_event_date is None or event_date < next_event_date):
            next_event_date = event_date

    artist_info['events'] = structured_events
    artist_info['next_event_date'] = next_event_date

In [11]:
artist_metadata_dict = {}

for artist in my_settings['artists']:
    # Step 1: build artist metadata
    building_artist_metadata(artist, artist_metadata_dict)

    # Step 2: fetch and attach upcoming events
    building_artist_event_data(artist, artist_metadata_dict)


ARTIS: Peter McPoland
ARTIS: Jonah Kagen
ARTIS: The Neighbourhood


In [12]:
artist_metadata_dict

{'Peter McPoland': {'ids': ['K8vZ917_hNf'],
  'genre_name': 'Alternative',
  'genre_id': 'KnvZfZ7vAvv',
  'ticketmaster_upcoming_events_number': 25,
  'pagination': {'page_size': 25, 'pages_needed': 1},
  'listening_score': None,
  'next_event_date': '2026-03-05',
  'total_notified_events': 0,
  'last_checked': '2025-11-16T18:13:45.394761',
  'events': [{'id': 'vv178ZbYGkMfj3HJ',
    'name': 'Peter McPoland: Big Lucky Tour',
    'date': '2026-03-05',
    'time': '18:00:00',
    'venue': 'House of Blues Chicago',
    'city': 'Chicago',
    'state': 'IL',
    'ticket_url': 'https://www.ticketmaster.com/peter-mcpoland-big-lucky-tour-chicago-illinois-03-05-2026/event/0400630DD4712F26',
    'status': 'onsale',
    'min_price': None,
    'max_price': None,
    'distance_miles': 122.00009607452122,
    'notified': False,
    'notification_count': 0,
    'notifications': []}]},
 'Jonah Kagen': {'ids': ['K8vZ917_0_7'],
  'genre_name': 'Pop',
  'genre_id': 'KnvZfZ7vAev',
  'ticketmaster_upcoming

In [ ]:
# Helper functions

In [ ]:
def compute_pagination(total_events, page_size=25):
    """
    Given a total number of events and a desired page size,
    compute how many pages are required.
    """
    # TODO: Add error handling (negative numbers, invalid types)
    
    pages_needed = math.ceil(total_events / page_size)
    
    return {
        "page_size": page_size,
        "pages_needed": pages_needed
    }


In [ ]:
def fetch_all_events(artist_id, travel_info, total_events):
    """
    Fetch all events for a given artist, using pagination.
    
    Returns a list of raw event dictionaries.
    """
    url = "https://app.ticketmaster.com/discovery/v2/events.json"
    page_size = 50  # Ticketmaster max
    pages_needed = (total_events + page_size - 1) // page_size

    all_events = []

    for page in range(pages_needed):
        params = {
            "attractionId": artist_id,
            "latlong": travel_info['latlong'],
            "radius": travel_info['radius'],
            "unit": travel_info['unit'],
            "apikey": get_api_key(),  # Load key only when making request
            "size": page_size,
            "page": page
        }

        # TODO: Add error handling
        response = requests.get(url, params=params)
        data = response.json()
        events = data.get("_embedded", {}).get("events", [])
        all_events.extend(events)

    return all_events

In [ ]:
# Core Functions

In [ ]:
def building_artist_metadata(artist_name, artist_dict):
    """
    Query Ticketmaster's Discovery API for a given artist and store key metadata.
    """

    # Base endpoint for searching artists
    url = "https://app.ticketmaster.com/discovery/v2/attractions.json"
    
    # Parameters for the API call
    params = {
        "keyword": artist_name,
        "apikey": get_api_key()  # Load key only when making request
    }

    # TODO: Add comprehensive error handling
    #  - Handle network failures
    #  - Handle non-200 responses
    #  - Handle missing or malformed JSON
    #  - Handle zero attractions returned
    #  - Handle multiple attraction matches
    
    # Execute API request and parse JSON
    artist_metadata = requests.get(url, params=params).json()
    
    # Extract the first attraction match
    # (Assuming at least one match exists for now)
    attraction = artist_metadata['_embedded']['attractions'][0]
    
    # Extract key upstream data
#     total_events = attraction['upcomingEvents']['ticketmaster']
    total_events = attraction['upcomingEvents']['_total']

    # Compute pagination info
    pagination_info = compute_pagination(total_events, page_size=25)

    # Build your structured metadata dictionary
    individual_artist_dict = {
        'id': attraction['id'],
        'genre_name': attraction['classifications'][0]['genre']['name'],
        'genre_id': attraction['classifications'][0]['genre']['id'],
        "ticketmaster_upcoming_events_number": total_events,
        "pagination": pagination_info
    }
    
    # Store metadata keyed by artist name
    artist_dict[artist_name] = individual_artist_dict

In [ ]:
def building_artist_event_data(artist_name, artist_dict):
    """
    Fetch and attach all upcoming events for a given artist in desired locations.
    """

    # Grab artist ID and total events
    artist_id = artist_dict[artist_name]['id']
    total_events = artist_dict[artist_name]['ticketmaster_upcoming_events_number']
    travel_info = my_settings['location']['travel_info']

    # Fetch all events using the pagination helper
    raw_events = fetch_all_events(artist_id, travel_info, total_events)

    # Transform raw events into your structured format
    structured_events = []
    for event in raw_events:
        event_data = {
            "id": event.get("id"),
            "name": event.get("name"),
            "date": event.get("dates", {}).get("start", {}).get("localDate"),
            "time": event.get("dates", {}).get("start", {}).get("localTime"),
            "venue": event.get("_embedded", {}).get("venues", [{}])[0].get("name"),
            "city": event.get("_embedded", {}).get("venues", [{}])[0].get("city", {}).get("name"),
            "state": event.get("_embedded", {}).get("venues", [{}])[0].get("state", {}).get("stateCode"),
            "ticket_url": event.get("url"),
            "notified": False,
            "notification_count": 0
        }
        structured_events.append(event_data)

    # Attach events to the artist dictionary
    artist_dict[artist_name]['events'] = structured_events

#     # Optional: timestamp
#     import datetime
#     artist_dict[artist_name]['last_updated'] = datetime.datetime.now().isoformat()


In [ ]:
# artist_metadata_dict = {}

# # add metadata
# for artist in my_settings['artists']:
#     building_artist_metadata(artist, artist_metadata_dict)

In [ ]:
# artist_metadata_dict

In [ ]:
# Adding in event data

In [ ]:
# artist_metadata_dict = {}

# for artist in my_settings['artists']:
#     # Step 1: build artist metadata
#     building_artist_metadata(artist, artist_metadata_dict)

#     # Step 2: fetch and attach upcoming events
#     building_artist_event_data(artist, artist_metadata_dict)


In [ ]:
# artist_metadata_dict